# Arabic ID-Card OCR — FastAPI v2
Responds with **full structured JSON** including all extracted OCR text per field.

In [1]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q fastapi uvicorn[standard] python-multipart pydantic pydantic-settings \
                ultralytics opencv-python-headless requests Pillow nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.4 MB/s eta 0:00:00


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ── 2. Write all API files ────────────────────────────────────────────────────
import os

API_DIR    = './api'
MODEL_DIR  = './saved_models'
os.makedirs(f'{API_DIR}/routers',  exist_ok=True)
os.makedirs(f'{API_DIR}/services', exist_ok=True)
os.makedirs(MODEL_DIR,             exist_ok=True)

OCR_SPACE_API_KEY = 'K86567801588957'   # ← your key

# ── config.py ────────────────────────────────────────────────────
with open(f'{API_DIR}/config.py', 'w') as f:
    f.write('''
import os
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    OCR_SPACE_API_KEY: str = "K86567801588957"
    REQUIRE_AUTH: bool = False
    MODEL_DIR: str = "../saved_models"
    ALLOWED_ORIGINS: list = ["*"]

    class Config:
        env_file = ".env"

settings = Settings()
''')

# ── auth.py ───────────────────────────────────────────────────────
with open(f'{API_DIR}/auth.py', 'w') as f:
    f.write('''
from fastapi import HTTPException
from config import settings

def verify_api_key(key=None):
    if not settings.REQUIRE_AUTH:
        return
    if key != settings.OCR_SPACE_API_KEY:
        raise HTTPException(status_code=401, detail="Invalid or missing API key")
''')

# ── services/ocr_service.py ───────────────────────────────────────
with open(f'{API_DIR}/services/ocr_service.py', 'w') as f:
    f.write('''
"""
ocr_service.py  —  YOLO + OCR.space pipeline
Returns a structured dict ready for JSON serialisation.
"""
import os, time, tempfile, requests
import cv2
import numpy as np
from ultralytics import YOLO
from config import settings

_model_cache: dict = {}

def _load_model(model_name):
    if model_name not in _model_cache:
        path = os.path.join(settings.MODEL_DIR, model_name)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model not found: {path}")
        _model_cache[model_name] = YOLO(path)
    return _model_cache[model_name]

def list_saved_models():
    d = settings.MODEL_DIR
    if not os.path.isdir(d):
        return {"models": []}
    return {"models": [f for f in os.listdir(d) if f.endswith(".pt")]}

def clear_cache(model_name=None):
    if model_name:
        _model_cache.pop(model_name, None)
    else:
        _model_cache.clear()


# ── Pre-processing ────────────────────────────────────────────────
def preprocess_crop(crop_bgr, label):
    h, w = crop_bgr.shape[:2]
    scale = 3 if (w < 200 or h < 40) else 2
    crop_bgr = cv2.resize(crop_bgr, (w * scale, h * scale),
                          interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.fastNlMeansDenoising(gray, h=10)
    lbl  = label.lower()
    if any(k in lbl for k in ("id", "number", "رقم")):
        _, binary = cv2.threshold(gray, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    elif any(k in lbl for k in ("date", "birth", "تاريخ")):
        binary = cv2.adaptiveThreshold(gray, 255,
                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    else:
        _, binary = cv2.threshold(gray, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        mean_val = np.mean(binary)
        if mean_val < 50 or mean_val > 200:
            binary = cv2.adaptiveThreshold(gray, 255,
                        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 15, 3)
    padded = cv2.copyMakeBorder(binary, 10, 10, 10, 10,
                                cv2.BORDER_CONSTANT, value=255)
    return cv2.cvtColor(padded, cv2.COLOR_GRAY2BGR)


# ── OCR.space call ────────────────────────────────────────────────
def _call_ocr_space(img_bgr):
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
        cv2.imwrite(tmp_path, img_bgr)
    try:
        with open(tmp_path, "rb") as f:
            resp = requests.post(
                "https://api.ocr.space/parse/image",
                files={"file": ("crop.jpg", f, "image/jpeg")},
                data={
                    "apikey":            settings.OCR_SPACE_API_KEY,
                    "language":          "ara",
                    "OCREngine":         2,
                    "isOverlayRequired": False,
                    "detectOrientation": True,
                    "scale":             True,
                },
                timeout=30,
            )
        result = resp.json()
        if result.get("IsErroredOnProcessing"):
            return ""
        parsed = result.get("ParsedResults", [])
        return parsed[0]["ParsedText"].strip() if parsed else ""
    except Exception as e:
        print(f"[OCR] error: {e}")
        return ""
    finally:
        os.unlink(tmp_path)


# ── Annotated image builder ───────────────────────────────────────
def _draw_annotations(image, detections):
    annotated = image.copy()
    colors = [(0,255,127),(0,180,255),(255,100,0),(180,0,255),(255,220,0)]
    for i, det in enumerate(detections):
        b   = det["box"]
        col = colors[i % len(colors)]
        cv2.rectangle(annotated, (b["x1"], b["y1"]), (b["x2"], b["y2"]), col, 2)
        label_txt = f\"{det[\'label\']}: {det[\'text\'][:30]}\" if det["text"] else det["label"]
        (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        ty = max(b["y1"] - 5, th + 4)
        cv2.rectangle(annotated, (b["x1"], ty-th-4), (b["x1"]+tw+4, ty+2), col, -1)
        cv2.putText(annotated, label_txt, (b["x1"]+2, ty-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 1)
    ok, buf = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 90])
    return buf.tobytes() if ok else b""


# ── Main pipeline ─────────────────────────────────────────────────
def run_pipeline(image_bytes, model_name="model.pt", conf=0.4, padding=8):
    """
    Returns:
      {
        "total": int,
        "detections": [{"id","label","confidence","text","box"}, ...],
        "ocr_json":   {label: text, ...},          # flat convenience dict
        "annotated_image_bytes": bytes
      }
    """
    nparr = np.frombuffer(image_bytes, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("Could not decode image.")

    img_h, img_w = image.shape[:2]
    img_area     = img_h * img_w

    model   = _load_model(model_name)
    results = model(image, conf=conf)[0]
    boxes   = results.boxes

    # filter bleed boxes (>50% card area)
    valid_indices = [
        i for i, xyxy in enumerate(boxes.xyxy)
        if ((int(xyxy[2])-int(xyxy[0])) * (int(xyxy[3])-int(xyxy[1]))) / img_area <= 0.50
    ]

    detections = []
    for idx, vi in enumerate(valid_indices):
        xyxy  = boxes.xyxy[vi]
        label = model.names[int(boxes.cls[vi])]
        conf_ = float(boxes.conf[vi])
        x1 = max(0,      int(xyxy[0]) - padding)
        y1 = max(0,      int(xyxy[1]) - padding)
        x2 = min(img_w,  int(xyxy[2]) + padding)
        y2 = min(img_h,  int(xyxy[3]) + padding)
        crop  = image[y1:y2, x1:x2]
        prep  = preprocess_crop(crop, label)
        text  = _call_ocr_space(prep)
        detections.append({
            "id":         idx + 1,
            "label":      label,
            "confidence": round(conf_, 4),
            "text":       text,
            "box":        {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
        })
        time.sleep(1.2)

    return {
        "total":                 len(detections),
        "detections":            detections,
        "ocr_json":              {d["label"]: d["text"] for d in detections},
        "annotated_image_bytes": _draw_annotations(image, detections),
    }
''')

# ── routers/health.py ─────────────────────────────────────────────
with open(f'{API_DIR}/routers/health.py', 'w') as f:
    f.write('''
from fastapi import APIRouter
from datetime import datetime

router = APIRouter()

@router.get("/health")
def health():
    return {"status": "ok", "timestamp": datetime.utcnow().isoformat(),
            "service": "Arabic ID-Card OCR API"}
''')

# ── routers/models.py ─────────────────────────────────────────────
with open(f'{API_DIR}/routers/models.py', 'w') as f:
    f.write('''
import os, shutil
from fastapi import APIRouter, File, UploadFile, HTTPException, Header
from typing import Optional
from services.ocr_service import list_saved_models, clear_cache
from config import settings
from auth import verify_api_key

router = APIRouter()

@router.get("/")
def list_models(x_api_key: Optional[str] = Header(None)):
    verify_api_key(x_api_key)
    return list_saved_models()

@router.post("/upload")
async def upload_model(file: UploadFile = File(...),
                       x_api_key: Optional[str] = Header(None)):
    verify_api_key(x_api_key)
    if not file.filename.endswith(".pt"):
        raise HTTPException(400, "Only .pt files accepted.")
    path = os.path.join(settings.MODEL_DIR, file.filename)
    with open(path, "wb") as f:
        shutil.copyfileobj(file.file, f)
    return {"message": f"{file.filename} saved",
            "size_mb": round(os.path.getsize(path)/1_048_576, 2)}

@router.delete("/{model_name}")
def delete_model(model_name: str, x_api_key: Optional[str] = Header(None)):
    verify_api_key(x_api_key)
    path = os.path.join(settings.MODEL_DIR, model_name)
    if not os.path.exists(path):
        raise HTTPException(404, "Not found.")
    os.remove(path)
    clear_cache(model_name)
    return {"message": f"{model_name} deleted"}

@router.post("/{model_name}/reload")
def reload_model(model_name: str, x_api_key: Optional[str] = Header(None)):
    verify_api_key(x_api_key)
    clear_cache(model_name)
    return {"message": f"{model_name} evicted from cache"}
''')

# ── routers/ocr.py ────────────────────────────────────────────────
with open(f'{API_DIR}/routers/ocr.py', 'w') as f:
    f.write('''
"""
routers/ocr.py  — All endpoints return full structured JSON.
"""
import base64
from fastapi import APIRouter, File, UploadFile, HTTPException, Query, Header
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
from typing import Optional, List, Dict
from services.ocr_service import run_pipeline
from auth import verify_api_key

router = APIRouter()

class Box(BaseModel):
    x1: int; y1: int; x2: int; y2: int

class Detection(BaseModel):
    id: int; label: str; confidence: float; text: str; box: Box

class OCRResponse(BaseModel):
    total:           int
    detections:      List[Detection]
    ocr_json:        Dict[str, str]   # flat dict  field_label -> extracted_text
    annotated_image: Optional[str] = None  # base64 JPEG


@router.post("/detect", response_model=OCRResponse)
async def detect(
    file:          UploadFile      = File(...),
    model_name:    str             = Query("model.pt"),
    conf:          float           = Query(0.4),
    padding:       int             = Query(8),
    include_image: bool            = Query(True),
    x_api_key:     Optional[str]   = Header(None),
):
    \"\"\"
    Run YOLO detection and OCR on an ID-card image.

    Returns full JSON including:
    - `total`      : number of detected fields
    - `detections` : list with label, confidence, extracted text, bounding box
    - `ocr_json`   : flat  { field_label: extracted_text }  for easy parsing
    - `annotated_image` : base64 JPEG (omit with include_image=false)

    Example `ocr_json`:
    ```json
    {
      "Name":       "محمد أحمد علي",
      "ID-Number":  "12345678901234",
      "Birth-Date": "01/01/1990",
      "Address":    "القاهرة - مصر"
    }
    ```
    \"\"\"
    verify_api_key(x_api_key)
    try:
        result = run_pipeline(await file.read(), model_name, conf, padding)
    except FileNotFoundError as e: raise HTTPException(404, str(e))
    except ValueError        as e: raise HTTPException(422, str(e))
    except Exception         as e: raise HTTPException(500, str(e))

    return OCRResponse(
        total      = result["total"],
        detections = result["detections"],
        ocr_json   = result["ocr_json"],
        annotated_image = (
            base64.b64encode(result["annotated_image_bytes"]).decode()
            if include_image and result["annotated_image_bytes"] else None
        ),
    )


@router.post("/detect/json")
async def detect_json_only(
    file:       UploadFile    = File(...),
    model_name: str           = Query("model.pt"),
    conf:       float         = Query(0.4),
    padding:    int           = Query(8),
    x_api_key:  Optional[str] = Header(None),
):
    """Same as /detect but never includes annotated image — compact JSON only."""
    verify_api_key(x_api_key)
    try:
        result = run_pipeline(await file.read(), model_name, conf, padding)
    except FileNotFoundError as e: raise HTTPException(404, str(e))
    except ValueError        as e: raise HTTPException(422, str(e))
    except Exception         as e: raise HTTPException(500, str(e))
    return JSONResponse({
        "total":      result["total"],
        "detections": result["detections"],
        "ocr_json":   result["ocr_json"],
    })


@router.post("/detect/image")
async def detect_image(
    file:       UploadFile    = File(...),
    model_name: str           = Query("model.pt"),
    conf:       float         = Query(0.4),
    padding:    int           = Query(8),
    x_api_key:  Optional[str] = Header(None),
):
    """Returns annotated JPEG only (no JSON text)."""
    verify_api_key(x_api_key)
    try:
        result = run_pipeline(await file.read(), model_name, conf, padding)
    except Exception as e:
        raise HTTPException(500, str(e))
    return Response(content=result["annotated_image_bytes"], media_type="image/jpeg")
''')

# ── __init__ files ────────────────────────────────────────────────
open(f'{API_DIR}/routers/__init__.py',  'w').close()
open(f'{API_DIR}/services/__init__.py', 'w').close()

# ── main.py ───────────────────────────────────────────────────────
with open(f'{API_DIR}/main.py', 'w') as f:
    f.write(f'''
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from routers import ocr, models, health
from config import settings

app = FastAPI(
    title="Arabic ID-Card OCR API",
    version="2.0.0",
    description="YOLO + OCR.space pipeline. Returns full structured JSON with extracted field text.",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=settings.ALLOWED_ORIGINS,
    allow_credentials=True, allow_methods=["*"], allow_headers=["*"],
)
app.include_router(health.router,               tags=["Health"])
app.include_router(ocr.router,    prefix="/ocr",    tags=["OCR"])
app.include_router(models.router, prefix="/models", tags=["Models"])
''')

# ── .env ──────────────────────────────────────────────────────────
with open(f'{API_DIR}/.env', 'w') as f:
    f.write(f'OCR_SPACE_API_KEY={OCR_SPACE_API_KEY}\nREQUIRE_AUTH=false\nMODEL_DIR=../saved_models\n')

# ── requirements.txt ──────────────────────────────────────────────
with open(f'{API_DIR}/requirements.txt', 'w') as f:
    f.write('fastapi>=0.111.0\nuvicorn[standard]>=0.30.0\npython-multipart>=0.0.9\n'
            'pydantic>=2.0.0\npydantic-settings>=2.0.0\nultralytics>=8.2.0\n'
            'opencv-python-headless>=4.9.0\nnumpy>=1.26.0\nrequests>=2.31.0\nPillow>=10.3.0\n')

print('✅ API v2 files written')
for root, _, files in os.walk(API_DIR):
    for fn in files:
        print(f'   {os.path.join(root, fn)}')

✅ API v2 files written
   ./api/auth.py
   ./api/requirements.txt
   ./api/config.py
   ./api/.env
   ./api/main.py
   ./api/routers/ocr.py
   ./api/routers/health.py
   ./api/routers/models.py
   ./api/routers/__init__.py
   ./api/services/ocr_service.py
   ./api/services/__init__.py


In [12]:
# ── 3. Copy your trained model ────────────────────────────────────────────────
import shutil, os
# !!! IMPORTANT: Update the 'src' path below to the actual path of your trained YOLO model (.pt file) !!!
src = '/python-apt.tar.xz' # e.g., '/content/yolov8_trained_model/weights/best.pt'
dst = './saved_models/model.pt'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'✅ Model copied → {dst}')
else:
    print('⚠️  Model not found at', src, '— update the path above to your actual trained model file.')


✅ Model copied → ./saved_models/model.pt


In [13]:
# ── 4. Start FastAPI server ───────────────────────────────────────────────────
import subprocess, threading, time, sys, os

API_PORT  = 8000
_api_proc = None

def start_api():
    global _api_proc
    env = os.environ.copy()
    env['PYTHONPATH'] = os.path.abspath('./api')
    _api_proc = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'main:app',
         '--host', '0.0.0.0', '--port', str(API_PORT)],
        cwd=os.path.abspath('./api'),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    def stream():
        for line in _api_proc.stdout:
            print('[API]', line.decode().rstrip())
    threading.Thread(target=stream, daemon=True).start()

start_api()
time.sleep(3)
print(f'\n✅ FastAPI running → http://localhost:{API_PORT}')
print(f'   Interactive docs → http://localhost:{API_PORT}/docs')


✅ FastAPI running → http://localhost:8000
   Interactive docs → http://localhost:8000/docs


In [19]:
# ── 5. Health check ───────────────────────────────────────────────────────────
import requests as req, json, time
BASE = 'http://localhost:8000'

# Implement a retry mechanism for the health check
max_retries = 5
retry_delay = 2 # seconds

for i in range(max_retries):
    try:
        r = req.get(f'{BASE}/health')
        r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        print('GET /health →', json.dumps(r.json(), indent=2))
        break # If successful, exit the loop
    except req.exceptions.ConnectionError:
        print(f'Connection refused. Retrying in {retry_delay} seconds... (Attempt {i+1}/{max_retries})')
        time.sleep(retry_delay)
    except req.exceptions.RequestException as e:
        print(f'Error during health check: {e}')
        break # Exit on other request-related errors
else:
    print(f'Failed to connect to FastAPI server after {max_retries} attempts.')


[API] INFO:     127.0.0.1:35416 - "GET /health HTTP/1.1" 200 OK
GET /health → {
  "status": "ok",
  "timestamp": "2026-03-05T13:12:30.753039",
  "service": "Arabic ID-Card OCR API"
}
